# 8.4 Lab: Inference Metrics and Goodput[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.4_inference_metrics/lab.ipynb)[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.4_inference_metrics/lab.ipynb)Build intuition for why averages lie, how goodput diverges from throughput under load,and why KV cache pressure is the leading indicator of SLO violations.

In [ ]:
# -- Cell 1: Install dependencies via subprocess --import subprocess, sys# Install numpy and matplotlib for numerical simulation and plottingsubprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib"])import numpy as np  # Numerical arrays and statisticsimport matplotlib.pyplot as plt  # Visualization library# Reproducible random state for consistent resultsnp.random.seed(42)# Clean visual style for all plotsplt.style.use('seaborn-v0_8-whitegrid')print("Setup complete.")

## Experiment 1: Percentile DistributionsTTFT follows a lognormal (prefill varies with prompt length).ITL follows a gamma (decode steps are regular with occasional interference spikes).We generate 1000 synthetic requests to show why p99 matters more than mean.

In [ ]:
# -- Cell 2: Generate synthetic request traces --# Number of simulated inference requestsn_requests = 1000# TTFT: lognormal distribution models prompt-length-dependent prefill time# Mean ~180ms with heavy right tail (long prompts cause high TTFT)ttft_values = np.random.lognormal(mean=np.log(180), sigma=0.6, size=n_requests)# ITL: gamma distribution models regular decode with occasional spikes# Shape=3 gives moderate variance; scale=15 centers around 45ms# Generate variable-length output per request (geometric distribution)output_lengths = np.random.geometric(p=0.02, size=n_requests)  # ~50 tokens avgoutput_lengths = np.clip(output_lengths, 5, 500)  # Bound between 5-500 tokens# Collect all ITL values across all requestsall_itl = []for length in output_lengths:    # Base ITL from gamma distribution    itl_base = np.random.gamma(shape=3.0, scale=15.0, size=length)    # 5% of decode steps get interference spikes (new prefills joining batch)    spike_mask = np.random.random(length) < 0.05    # Spikes multiply latency by 2-5x    itl_base[spike_mask] *= np.random.uniform(2, 5, size=spike_mask.sum())    all_itl.extend(itl_base)# Convert to numpy array for percentile computationall_itl = np.array(all_itl)# Compute percentiles for both metricspercentiles = [50, 75, 90, 95, 99]ttft_pcts = np.percentile(ttft_values, percentiles)itl_pcts = np.percentile(all_itl, percentiles)# Display the percentile tableprint("TTFT Percentiles:")for p, v in zip(percentiles, ttft_pcts):    print(f"  p{p:02d}: {v:>8.1f} ms")print(f"  mean: {ttft_values.mean():>7.1f} ms")print(f"  p99/p50 ratio: {ttft_pcts[4]/ttft_pcts[0]:.1f}x")print(f"\nITL Percentiles:")for p, v in zip(percentiles, itl_pcts):    print(f"  p{p:02d}: {v:>8.1f} ms")print(f"  mean: {all_itl.mean():>7.1f} ms")print(f"  p99/p50 ratio: {itl_pcts[4]/itl_pcts[0]:.1f}x")

In [ ]:
# -- Cell 3: Visualize distributions with percentile markers --# Create side-by-side histograms for TTFT and ITLfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Color palette for percentile lines (matches mermaid diagram colors)pct_colors = ["#2563eb", "#7c3aed", "#dc2626", "#ea580c", "#0d9488"]# Left plot: TTFT distributionaxes[0].hist(ttft_values, bins=60, alpha=0.7, color="#dbeafe", edgecolor="#1e293b")# Draw vertical lines at each percentile to show tail behaviorfor p, v, c in zip(percentiles, ttft_pcts, pct_colors):    axes[0].axvline(v, color=c, linestyle="--", linewidth=1.5, label=f"p{p}={v:.0f}ms")axes[0].set_xlabel("TTFT (ms)")axes[0].set_ylabel("Count")axes[0].set_title("TTFT Distribution: Averages Hide the Tail")axes[0].legend(fontsize=9)# Right plot: ITL distributionaxes[1].hist(all_itl, bins=60, alpha=0.7, color="#dcfce7", edgecolor="#1e293b")# Same percentile markers for ITLfor p, v, c in zip(percentiles, itl_pcts, pct_colors):    axes[1].axvline(v, color=c, linestyle="--", linewidth=1.5, label=f"p{p}={v:.0f}ms")axes[1].set_xlabel("ITL (ms)")axes[1].set_ylabel("Count")axes[1].set_title("ITL Distribution: Spikes from Batch Interference")axes[1].legend(fontsize=9)plt.tight_layout()plt.show()# The gap between p50 and p99 reveals the true worst-case user experienceprint(f"TTFT: mean={ttft_values.mean():.0f}ms but p99={ttft_pcts[4]:.0f}ms (hidden by averages)")

## Experiment 2: Goodput vs Raw Throughput Under LoadAs load increases, raw throughput climbs (more tokens generated) but goodput peaksthen drops (increasing fraction of tokens violate SLOs). The divergence point isyour actual capacity limit.

In [ ]:
# -- Cell 4: Simulate goodput divergence under increasing load --# Define SLO constraintsslo_ttft_ms = 500.0   # Maximum acceptable TTFTslo_itl_ms = 100.0    # Maximum acceptable per-token ITL# Sweep load from 50% to 300% of baseline capacityload_multipliers = np.linspace(0.5, 3.0, 25)# Storage for throughput metrics at each load levelraw_throughput_values = []goodput_values = []for load_mult in load_multipliers:    # Under higher load, queue wait increases (queuing theory)    # TTFT grows super-linearly near capacity    load_factor = 1.0 + (load_mult - 1.0) ** 1.5 * 0.8    # Simulate 200 requests at this load level    sim_ttft = ttft_values[:200] * load_factor + np.random.exponential(20 * load_mult, 200)    sim_itl_means = np.random.gamma(3, 15 * load_factor, 200)    # Raw throughput: total tokens / time window    total_tokens = output_lengths[:200].sum()    # Time window shrinks with higher arrival rate    time_window = 200 / (10 * load_mult)  # 10 req/s baseline    raw_tps = total_tokens / time_window    # Goodput: only count tokens from SLO-meeting requests    good_tokens = 0    for i in range(200):        # Request meets SLO only if BOTH TTFT and ITL are within bounds        ttft_ok = sim_ttft[i] <= slo_ttft_ms        itl_ok = sim_itl_means[i] <= slo_itl_ms        if ttft_ok and itl_ok:            good_tokens += output_lengths[i]    # Goodput in tokens/second    good_tps = good_tokens / time_window    raw_throughput_values.append(raw_tps)    goodput_values.append(good_tps)# Plot the divergencefig, ax = plt.subplots(figsize=(10, 5))# Raw throughput keeps climbing (system is busy)ax.plot(load_multipliers, raw_throughput_values, "b-o", markersize=4, label="Raw Throughput (tok/s)")# Goodput peaks then falls (SLO violations increase)ax.plot(load_multipliers, goodput_values, "r-s", markersize=4, label="Goodput (good tok/s)")# Shade the waste region between raw and goodax.fill_between(load_multipliers, goodput_values, raw_throughput_values,                alpha=0.15, color="red", label="Wasted tokens (SLO violations)")# Mark baseline loadax.axvline(1.0, color="gray", linestyle=":", label="Baseline load (1.0x)")# Find and mark optimal operating point (max goodput)optimal_idx = np.argmax(goodput_values)ax.scatter([load_multipliers[optimal_idx]], [goodput_values[optimal_idx]],           color="green", s=100, zorder=5, label=f"Optimal: {load_multipliers[optimal_idx]:.1f}x load")ax.set_xlabel("Load Multiplier (1.0 = baseline capacity)")ax.set_ylabel("Tokens / Second")ax.set_title("Raw Throughput vs Goodput: The Capacity Illusion")ax.legend()plt.tight_layout()plt.show()# Key insight: raw throughput keeps climbing but useful work peaks then dropsprint(f"Optimal operating point: {load_multipliers[optimal_idx]:.1f}x baseline load")print(f"Beyond this, adding more requests REDUCES useful output.")

## Experiment 3: KV Cache Pressure and PreemptionWhen KV cache occupancy exceeds ~85%, the scheduler must preempt (evict) sequences.Each preemption destroys partial work and spikes TTFT for re-queued requests.

In [ ]:
# -- Cell 5: Simulate KV cache occupancy with preemption events --# Simulation parametersduration_seconds = 60  # Simulate 60 seconds of operationdt = 0.1  # 100ms time resolutiontimesteps = np.arange(0, duration_seconds, dt)# KV cache state trackingkv_occupancy = np.zeros_like(timesteps)preemption_times = []  # Track when preemptions occurpreemption_occupancy = []  # Track occupancy at preemption# Alert threshold: preemptions happen above this levelalert_threshold = 0.85# Initial occupancy starts at 40% (some requests already running)current_occupancy = 0.40for i, t in enumerate(timesteps):    # Occupancy grows as new requests arrive and extend their KV caches    # Growth rate: ~1.2% per second (new tokens being generated)    growth = 0.012 * dt    # Burst arrivals cause periodic spikes    burst = 0.003 * np.sin(t * 0.5) * dt    # Random noise from variable request sizes    noise = np.random.normal(0, 0.005)    # Update occupancy    current_occupancy += growth + burst + noise    # Preemption check: if over threshold, scheduler evicts sequences    if current_occupancy > alert_threshold:        preemption_times.append(t)        preemption_occupancy.append(current_occupancy)        # Eviction drops occupancy by ~15% (several sequences removed)        current_occupancy -= 0.15    # Clamp to valid range    current_occupancy = np.clip(current_occupancy, 0, 1.0)    kv_occupancy[i] = current_occupancy# Visualize the KV cache pressure over timefig, ax = plt.subplots(figsize=(12, 5))# Main occupancy lineax.plot(timesteps, kv_occupancy * 100, color="#2563eb", linewidth=1.5, label="KV Cache Occupancy")# Alert threshold lineax.axhline(alert_threshold * 100, color="#dc2626", linestyle="--", linewidth=2,           label=f"Alert Threshold ({alert_threshold*100:.0f}%)")# Mark preemption events as red dotsif preemption_times:    ax.scatter(preemption_times, [p * 100 for p in preemption_occupancy],               color="red", s=50, zorder=5, label=f"Preemptions ({len(preemption_times)})")# Labelsax.set_xlabel("Time (seconds)")ax.set_ylabel("KV Cache Occupancy (%)")ax.set_title("KV Cache Pressure: Each Spike Triggers Preemption (Wasted Work)")ax.set_ylim(0, 100)ax.legend()plt.tight_layout()plt.show()# Each red dot = destroyed partial generation + re-queued request + TTFT spikeprint(f"Preemption events in 60s: {len(preemption_times)}")print(f"Each event destroys partial work and spikes TTFT for affected requests.")

## Experiment 4: Cost Per Million Good TokensThe U-curve of inference cost: too-low utilization wastes money on idle GPUs,too-high utilization wastes money on SLO-violating tokens that do not count.

In [ ]:
# -- Cell 6: Model cost efficiency vs utilization --# GPU pricing parametersgpu_cost_per_hour = 3.50  # A100 80GB on-demand price# Maximum achievable throughput at full utilizationmax_throughput_tps = 2000.0  # Peak tokens/sec for this GPU# Sweep utilization from 10% to 99%utilization_range = np.linspace(0.10, 0.99, 50)# Goodput ratio model: stays high until ~70% util, then drops# This models the real behavior where overload causes SLO violationsgoodput_ratios = np.where(    utilization_range <= 0.70,    # Below 70% util: goodput is nearly 100% (no queuing pressure)    0.98 - 0.02 * (0.70 - utilization_range),    # Above 70%: goodput drops quadratically (queuing builds fast)    0.98 - 2.0 * (utilization_range - 0.70) ** 2)# Clamp goodput ratio to reasonable boundsgoodput_ratios = np.clip(goodput_ratios, 0.4, 0.99)# Compute cost per million tokens at each utilization level# Raw cost: what you pay per million tokens generated (including wasted)raw_cost_per_M = []# Effective cost: what you pay per million GOOD tokenseffective_cost_per_M = []for util, gp_ratio in zip(utilization_range, goodput_ratios):    # Tokens generated per hour at this utilization    tokens_per_hour = max_throughput_tps * util * 3600    # Raw cost per million tokens    raw_cpm = (gpu_cost_per_hour / tokens_per_hour) * 1_000_000    raw_cost_per_M.append(raw_cpm)    # Effective cost divides by goodput ratio (paying more per useful token)    effective_cpm = raw_cpm / gp_ratio    effective_cost_per_M.append(effective_cpm)# Find the optimal utilization (minimum effective cost)optimal_idx = np.argmin(effective_cost_per_M)optimal_util = utilization_range[optimal_idx]optimal_cost = effective_cost_per_M[optimal_idx]# Plot the U-curvefig, ax = plt.subplots(figsize=(10, 5))ax.plot(utilization_range * 100, raw_cost_per_M, "b-", linewidth=2, label="Raw $/M tokens")ax.plot(utilization_range * 100, effective_cost_per_M, "r--", linewidth=2, label="Effective $/M good tokens")# Mark optimal pointax.scatter([optimal_util * 100], [optimal_cost], color="green", s=120, zorder=5,           label=f"Optimal: {optimal_util*100:.0f}% util = ${optimal_cost:.3f}/M")# Shade the waste zonesax.axvspan(0, 40, alpha=0.05, color="blue", label="Idle waste zone")ax.axvspan(80, 100, alpha=0.05, color="red", label="Overload waste zone")ax.set_xlabel("GPU Utilization (%)")ax.set_ylabel("Cost ($ / Million Tokens)")ax.set_title("Cost U-Curve: Idle Waste vs Overload Waste")ax.legend(loc="upper right")ax.set_ylim(0, max(raw_cost_per_M[:5]) * 1.2)plt.tight_layout()plt.show()# The optimal point balances idle cost against SLO violation costprint(f"Optimal utilization: {optimal_util*100:.0f}%")print(f"Effective cost at optimal: ${optimal_cost:.3f}/M good tokens")print(f"At 95% util, effective cost: ${effective_cost_per_M[-3]:.3f}/M (overload penalty)")

## Key Takeaways1. **p99/p50 ratio of 10-50x** is normal for LLM inference (vs 2-5x for web services)2. **Goodput peaks then drops** as load increases: raw throughput is a vanity metric3. **Any preemption rate > 0** signals capacity exhaustion requiring immediate action4. **Cost has a U-curve**: optimal utilization is typically 65-75% (not 95%)5. **Open-loop benchmarking is mandatory**: closed-loop hides 3-10x worse tail latency